# Olist E-commerce — Data Preparation

Prepare nine Olist source datasets for MySQL 8+, while preserving the project’s original transformation rules.

**Run location:** open this notebook from `notebooks/` and run with the working directory set to that folder. The code reads `../data/raw/` and writes `../data/cleaned/`.

**Outputs:** customer, order, order-item, payment, product, category-translation, seller, review, and geolocation files, plus a MySQL-safe review TSV. This notebook does not generate Tableau exports; those are produced by the SQL scripts.

Required package: `pandas`. The original Olist CSV files must be in `data/raw/`.

In [ ]:
from pathlib import Path
import csv
import pandas as pd

RAW = Path('../data/raw')
CLEAN = Path('../data/cleaned')
CLEAN.mkdir(parents=True, exist_ok=True)

assert RAW.is_dir(), 'Run notebook from notebooks/ with data/raw/ present.'
print('Source directory:', RAW.resolve())
print('Output directory:', CLEAN.resolve())

## 1. Customers

Preserve `customer_id` as the order-specific identifier and `customer_unique_id` for repeat-customer analysis. Standardize ZIP prefixes and place names.

In [ ]:
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
print("Input rows:", len(customers), "| Duplicate rows:", customers.duplicated().sum())

In [ ]:
#clean the customers table

customers_clean = customers.copy()

In [ ]:
#standardize the Zip-code prefix

customers_clean["customer_zip_code_prefix"] = (
    customers_clean["customer_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

In [ ]:
#Standardize the customer city and state text

customers_clean["customer_city"] = (
    customers_clean["customer_city"]
    .str.strip()
    .str.lower()
)

customers_clean["customer_state"] = (
    customers_clean["customer_state"]
    .str.strip()
    .str.upper()
)

In [ ]:
assert customers_clean['customer_id'].is_unique
assert customers_clean['customer_zip_code_prefix'].str.len().eq(5).all()
customers_clean.to_csv(CLEAN / 'customers_clean.csv', index=False)

## 2. Orders

Convert dates without dropping missing fulfillment dates; not every order is delivered. Standardize order statuses.

In [ ]:
orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
print("Input rows:", len(orders), "| Duplicate rows:", orders.duplicated().sum())

In [ ]:
#creating a cleaned copy

orders_clean = orders.copy()

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date" 
]

In [ ]:
#converting the columns to datetime format

for column in date_columns:
    orders_clean[column] = pd.to_datetime(orders_clean[column], errors='coerce')

In [ ]:
#standardizing the order status text

orders_clean["order_status"] = (
    orders_clean["order_status"]
    .str.strip()
    .str.lower()
)

In [ ]:
assert orders_clean['order_id'].is_unique
orders_clean.to_csv(CLEAN / 'orders_clean.csv', index=False, date_format='%Y-%m-%d %H:%M:%S')

## 3. Order items

Keep `(order_id, order_item_id)` as the composite key, and convert the shipping-limit timestamp.

In [ ]:
order_items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
print("Input rows:", len(order_items), "| Duplicate rows:", order_items.duplicated().sum())

In [ ]:
#creating a cleaned copy of the order_items dataset
order_items_clean = order_items.copy()

In [ ]:
#converting the data column
order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"], errors='coerce'
)

In [ ]:
assert not order_items_clean.duplicated(['order_id', 'order_item_id']).any()
order_items_clean.to_csv(CLEAN / 'order_items_clean.csv', index=False, date_format='%Y-%m-%d %H:%M:%S')

## 4. Payments

Preserve multiple payment records per order; standardize payment-type text.

In [ ]:
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
print("Input rows:", len(payments), "| Duplicate rows:", payments.duplicated().sum())

In [ ]:
#creating a cleaned copy of the payments dataset
payments_clean = payments.copy()

In [ ]:
#standardizing the payment_type 

payments_clean["payment_type"] = (
    payments_clean["payment_type"]
    .str.strip()
    .str.lower()    
)

In [ ]:
assert not payments_clean.duplicated(['order_id','payment_sequential']).any()
payments_clean.to_csv(CLEAN / 'payments_clean.csv', index=False)

## 5. Products

Correct the source’s misspelled length columns, standardize missing categories, and retain nullable counts.

In [ ]:
products = pd.read_csv(RAW / "olist_products_dataset.csv")
print("Input rows:", len(products), "| Duplicate rows:", products.duplicated().sum())

In [ ]:
#creating a cleaned copy of the products dataset
products_clean = products.copy()

In [ ]:
products_clean = products_clean.rename(
    columns = {
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length"}
)

In [ ]:
#Handling missing categories

products_clean["product_category_name"] = (
    products_clean["product_category_name"].fillna("unknown")
    .str.strip()
    .str.lower()
)

In [ ]:
#covert count columns to nullable integers

count_columns = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty"
]

for column in count_columns:
    products_clean[column] = products_clean[column].astype("Int64")

In [ ]:
#converting zero weights into missing values

products_clean.loc[
    products_clean["product_weight_g"] == 0, "product_weight_g"
] = pd.NA

In [ ]:
assert products_clean['product_id'].is_unique
products_clean.to_csv(CLEAN / 'products_clean.csv', index=False)

## 6. Category translation

Clean category strings and preserve the two additional translations identified in the original notebook.

In [ ]:
category_translation = pd.read_csv(RAW / "product_category_name_translation.csv")
print("Input rows:", len(category_translation), "| Duplicate rows:", category_translation.duplicated().sum())

In [ ]:
#creating a cleaned copy of the category_translation dataset
category_translation_clean = category_translation.copy()

In [ ]:
#clean the portuguese category names
category_translation_clean["product_category_name"] = (
    category_translation_clean["product_category_name"]
    .str.strip()
    .str.lower()
)

In [ ]:
#clean the english category names
category_translation_clean["product_category_name_english"] = (
    category_translation_clean["product_category_name_english"]
    .str.strip()
    .str.lower()    
)

In [ ]:
#creating missed translation

missing_translations = pd.DataFrame(
    {
        "product_category_name": [
            "pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos" ],

        "product_category_name_english": [
        "pc_gaming", "portable_kitchen_and_food_preparers"
        ]    
    }
)

In [ ]:
#adding them to the cleaned translation table

category_translation_clean = pd.concat(
    [
        category_translation_clean,
        missing_translations
    ],
    ignore_index=True
)

In [ ]:
assert category_translation_clean['product_category_name'].is_unique
category_translation_clean.to_csv(CLEAN / 'category_translation_clean.csv', index=False)

## 7. Sellers

Standardize sellers’ ZIP-code prefixes, city names, and state abbreviations.

In [ ]:
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")
print("Input rows:", len(sellers), "| Duplicate rows:", sellers.duplicated().sum())

In [ ]:
#creating a cleaned copy of the sellers dataset
sellers_clean = sellers.copy()

In [ ]:
#covert the zip code prefix to string and standardize to 5 digits

sellers_clean["seller_zip_code_prefix"] = (
    sellers_clean["seller_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

In [ ]:
#standardize city names

sellers_clean["seller_city"] = (
    sellers_clean["seller_city"]
    .str.strip()
    .str.lower()
)

In [ ]:
#standardize state abbreviations

sellers_clean["seller_state"] = (
    sellers_clean["seller_state"]
    .str.strip()    
    .str.upper()
)

In [ ]:
assert sellers_clean['seller_id'].is_unique
sellers_clean.to_csv(CLEAN / 'sellers_clean.csv', index=False)

## 8. Reviews

Convert review timestamps and normalize blank comment fields. Also produce a tab-separated import file with single-line text to avoid MySQL ingestion problems.

In [ ]:
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
print("Input rows:", len(reviews), "| Duplicate rows:", reviews.duplicated().sum())

In [ ]:
#create a cleaned copy

reviews_clean = reviews.copy()

In [ ]:
review_data_columns = [
    "review_creation_date",
    "review_answer_timestamp",
]

In [ ]:
#convert the dates

for column in review_data_columns:
    reviews_clean[column] = pd.to_datetime(
        reviews_clean[column], errors='coerce'
    )

In [ ]:
#cleaning review titles

reviews_clean["review_comment_title"] = (
    reviews_clean["review_comment_title"]
    .str.strip()
)

In [ ]:
#cleaning review messages

reviews_clean["review_comment_message"] = (
    reviews_clean["review_comment_message"]
    .str.strip()
)

In [ ]:
#converting blank strings into missing values

reviews_clean["review_comment_title"] = (
    reviews_clean["review_comment_title"].replace("", pd.NA)
)

reviews_clean["review_comment_message"] = (
    reviews_clean["review_comment_message"].replace("", pd.NA)
)

In [ ]:
assert not reviews_clean.duplicated(['review_id','order_id']).any()
reviews_clean.to_csv(CLEAN / 'reviews_clean.csv', index=False, date_format='%Y-%m-%d %H:%M:%S')

reviews_mysql = reviews_clean.copy()
for column in ['review_comment_title', 'review_comment_message']:
    reviews_mysql[column] = (
        reviews_mysql[column].astype('string')
        .str.replace(r'[\r\n\t]+', ' ', regex=True)
        .str.strip()
    )
reviews_mysql.to_csv(
    CLEAN / 'reviews_mysql.tsv', sep='\t', index=False, encoding='utf-8',
    quoting=csv.QUOTE_MINIMAL, quotechar='"', doublequote=True,
    lineterminator='\n', date_format='%Y-%m-%d %H:%M:%S'
)
check = pd.read_csv(CLEAN / 'reviews_mysql.tsv', sep='\t')
assert len(check) == len(reviews_mysql)
assert not check.duplicated(['review_id','order_id']).any()

## 9. Geolocation

Remove exact duplicate location records and create **one representative record per ZIP prefix** using median coordinates and modal city/state. Retain detailed cleaned data for audit.

In [ ]:
geolocation = pd.read_csv(RAW / "olist_geolocation_dataset.csv")
print("Input rows:", len(geolocation), "| Duplicate rows:", geolocation.duplicated().sum())

In [ ]:
geolocation_clean = (
    geolocation
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

In [ ]:
#convert zip prefixes to five-character text

geolocation_clean[
    "geolocation_zip_code_prefix"] = (
        geolocation_clean["geolocation_zip_code_prefix"]
        .astype(str)
        .str.zfill(5)
    )

In [ ]:
#standardize state abbrevations

geolocation_clean["geolocation_state"] = (
    geolocation_clean["geolocation_state"]
    .str.strip()
    .str.upper()
)

In [ ]:
#creating a mode function

def most_common_values(series):
    modes = series.mode()

    if len(modes) > 0:
        return modes.iloc[0]

    return pd.NA

In [ ]:
#Group by ZIP code and summarize

geolocation_summary = (
    geolocation_clean.groupby(
        "geolocation_zip_code_prefix",
        as_index=False
    )
    .agg(
        geolocation_lat=(
            "geolocation_lat",
            "median"
    ),
    geolocation_lng = (
        "geolocation_lng",
        "median"
        ),
    geolocation_city=(
        "geolocation_city",
        most_common_values
    ),
    geolocation_state = (
        "geolocation_state",
        most_common_values
    ),
    geolocation_observation_count=(
        "geolocation_zip_code_prefix",
        "size"
    )
    )
)

In [ ]:
assert geolocation_summary['geolocation_zip_code_prefix'].is_unique
assert geolocation_summary['geolocation_zip_code_prefix'].str.len().eq(5).all()
geolocation_clean.to_csv(CLEAN / 'geolocation_clean.csv', index=False)
geolocation_summary.to_csv(CLEAN / 'geolocation_summary.csv', index=False, encoding='utf-8')

## 10. Cross-table checks

Confirm key relationships before loading into MySQL. Missing optional child rows (for example, an order without a review) should be documented rather than automatically deleted.

In [ ]:
checks = {
    'orders_missing_customers': len(set(orders_clean.customer_id) - set(customers_clean.customer_id)),
    'items_missing_orders': len(set(order_items_clean.order_id) - set(orders_clean.order_id)),
    'items_missing_products': len(set(order_items_clean.product_id) - set(products_clean.product_id)),
    'items_missing_sellers': len(set(order_items_clean.seller_id) - set(sellers_clean.seller_id)),
    'payments_missing_orders': len(set(payments_clean.order_id) - set(orders_clean.order_id)),
    'reviews_missing_orders': len(set(reviews_clean.order_id) - set(orders_clean.order_id)),
    'orders_without_items': len(set(orders_clean.order_id) - set(order_items_clean.order_id)),
    'orders_without_payments': len(set(orders_clean.order_id) - set(payments_clean.order_id)),
    'orders_without_reviews': len(set(orders_clean.order_id) - set(reviews_clean.order_id)),
}
pd.Series(checks, name='row_count').to_frame()

## 11. Export summary

`geolocation_summary.csv` (not the detailed `geolocation_clean.csv`) is loaded into MySQL. MySQL uses `reviews_mysql.tsv` for review imports. See `sql/01_database_setup.sql` for the database schema and import order.

In [ ]:
for name in [
    'customers_clean.csv', 'orders_clean.csv', 'order_items_clean.csv',
    'payments_clean.csv', 'products_clean.csv', 'category_translation_clean.csv',
    'sellers_clean.csv', 'reviews_clean.csv', 'reviews_mysql.tsv',
    'geolocation_clean.csv', 'geolocation_summary.csv',
]:
    file = CLEAN / name
    assert file.exists(), f'Missing output: {file}'
    print(f'{name}: {file.stat().st_size / 1024 / 1024:.2f} MB')